# 🎙️ Nepal RAG + SLM + ASR — Unified FastAPI Server

Combines the English ASR pipeline with the RAG + SLM pipeline into a single FastAPI server exposed via ngrok.

## How it works

| Input sent to `/ask` | What happens |
|---|---|
| **Text only** (`question` field) | RAG retrieval → SLM → returns `rag_answer` + `sources` |
| **Audio file** (`.wav`) | ASR transcription → RAG retrieval → SLM → returns `transcription` + `rag_answer` + `sources` |

When audio is sent the server:
1. Saves the uploaded audio to a temp file
2. Transcribes it with the English FastConformer ASR model
3. Uses the transcription as the question for RAG + SLM
4. Returns **both** the transcription and the RAG answer

When no audio is sent, only the RAG answer is returned (transcription field is `null`).

## Endpoints

| Method | Path | Description |
|---|---|---|
| `GET` | `/health` | Server + GPU + model status |
| `POST` | `/ask` | JSON question → RAG answer |
| `POST` | `/ask_audio` | Audio file upload → transcription + RAG answer |
| `GET` | `/docs` | Auto-generated Swagger UI |

## Setup steps
1. Select **GPU** runtime: `Runtime → Change runtime type → T4 GPU`
2. Add your ngrok token to Colab Secrets: `Tools → Secrets → Add` → name it `ngrok_auth`
3. Verify the model paths in **Section 3** match your Drive layout
4. Run all cells top to bottom
5. If dependencies were reinstalled/changed, **restart runtime once** and re-run all cells
6. Copy the printed public URL

## Dependency stability note
- This notebook pins a compatible set for ASR + RAG (`numpy==1.26.4`, `scipy==1.11.4`, `scikit-learn==1.4.2`, `transformers==4.41.2`, `sentence-transformers==2.7.0`).
- Mixing newer NumPy with older NeMo/Torch can trigger: `ASR transcription failed: Numpy is not available`.
- Mismatched SciPy/sklearn/transformers can surface as embedder/reranker input errors.

## ⚙️ Section 1 — Mount Drive & Clone Repo

In [1]:
# ── Cell 0 : Mount Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/drive')
print('✅ Drive mounted')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).
✅ Drive mounted


In [2]:
# ── Cell 1 : Clone RAG + SLM repo ────────────────────────────
import os, sys

BRANCH   = 'RAG-Path'
REPO_URL = 'https://github.com/ShreejayShakya28/ASR-LLM-Pipeline.git'
REPO_DIR = '/content/ASR-LLM-Pipeline'
RAG_DIR  = f'{REPO_DIR}/RAG'
SLM_DIR  = f'{REPO_DIR}/SLM'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR} -q

for d in [RAG_DIR, SLM_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

print('✅ Repo ready')

✅ Repo ready


In [3]:
# ── Cell 2a : Install RAG requirements ───────────────────────
!pip install -q -r /content/ASR-LLM-Pipeline/RAG/requirements.txt
print('✅ RAG requirements installed')

✅ RAG requirements installed


In [4]:
# ── Cell 2b : Install NeMo ASR (English FastConformer) ───────
# NeMo is large — this takes ~3-5 minutes on first run.
# The [asr] extra installs only the ASR sub-collection, not the full toolkit.
!pip install -q 'nemo_toolkit[asr]'
print('✅ NeMo ASR installed')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
blosc2 4.1.2 requires numexpr>=2.14.1; platform_machine != "wasm32", but you have numexpr 2.13.1 which is incompatible.
db-dtypes 1.5.0 requires panda

In [ ]:
# ── Cell 2b.1 : Pin compatible ML stack (ASR + RAG) ───────────
# Why: NeMo ASR can fail with NumPy 2.x in some runtimes, while
# embedder/reranker can fail when SciPy/sklearn/transformers drift.
# This lock set is chosen to keep both ASR and RAG stable together.

!pip install -q --upgrade \
  "numpy==1.26.4" \
  "scipy==1.11.4" \
  "scikit-learn==1.4.2" \
  "transformers==4.41.2" \
  "sentence-transformers==2.7.0"

print('✅ Dependency versions pinned for ASR + RAG compatibility')
print('⚠️ If this is the first run after package changes, restart runtime once and run all cells again.')

In [5]:
# ── Cell 2c : Install FastAPI + uvicorn + ngrok ───────────────
!pip install -q fastapi 'uvicorn[standard]' python-multipart pyngrok
print('✅ FastAPI + uvicorn + pyngrok installed')

✅ FastAPI + uvicorn + pyngrok installed


## 🔑 Section 2 — ngrok Auth Token

Add your token in Colab: `Tools → Secrets → Add new secret`  
Name it **`ngrok_auth`**, paste your token from https://dashboard.ngrok.com/get-started/your-authtoken


In [6]:
# ── Cell 3 : ngrok auth ───────────────────────────────────────
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('rag_auth')

# ── Kaggle alternative (comment the two lines above, uncomment below) ──
# from kaggle_secrets import UserSecretsClient
# NGROK_AUTH_TOKEN = UserSecretsClient().get_secret('ngrok_auth')

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('✅ ngrok configured')

✅ ngrok configured


## 🤖 Section 3 — Load All Models

Three models are loaded once at startup and reused across all requests:

| Model | Purpose | Loaded from |
|---|---|---|
| `embedding_model` | FAISS vector search | HuggingFace (auto-download) |
| `reranker` | CrossEncoder rerank | HuggingFace (auto-download) |
| `slm_model` | Answer generation | Google Drive |
| `asr_model` | English audio → text | Google Drive |

**Verify these Drive paths match your setup before running:**
- `SLM_MODEL_PATH` — your SLM `.pth` file
- `ASR_MODEL_PATH` — your FastConformer `.nemo` file
- `ASR_LM_PATH` — your n-gram LM `.nemo` file (set to `None` to disable)


In [9]:
pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-metrics 4.0.0 requires numpy>=2.2.2, but you have numpy 1.26.4 which is incompatible.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires nu

In [12]:
import sys
try:
    import numpy
    print(f"Numpy Version: {numpy.__version__}")
except ImportError:
    print("Numpy is not currently importable. Please go to 'Runtime' -> 'Restart runtime' to apply the installation changes.")
except Exception as e:
    print(f"Error: {e}")

Numpy Version: 1.26.4


In [13]:
from rag.models import embedding_model, reranker
print('✅ Embedder and reranker ready')

📦 Loading embedding model...
   ✅ all-MiniLM-L6-v2
📦 Loading reranker...
   ✅ cross-encoder/ms-marco-MiniLM-L-6-v2

✅ Retrieval models ready.
✅ Embedder and reranker ready


In [14]:
# ── Cell 4b : Load SLM ────────────────────────────────────────
import torch
from inference import load_model

# ✏️  Update this path if your SLM is stored elsewhere on Drive
SLM_MODEL_PATH = '/drive/MyDrive/RAG/SLM_Model/SLM.pth'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

slm_model, slm_config = load_model(SLM_MODEL_PATH, 'gpt2-medium (355M)', DEVICE)
print(f'✅ SLM loaded on {DEVICE}')

✅ SLM loaded on cuda


In [15]:
# ── Cell 4c : Load English ASR (FastConformer) ────────────────
import nemo.collections.asr as nemo_asr

# ✏️  Update these paths to match your Drive layout
ASR_MODEL_PATH = '/drive/MyDrive/asr_sangam/fastconformer.nemo'
ASR_LM_PATH    = None  # set to None to disable LM

asr_model = nemo_asr.models.EncDecCTCModelBPE.restore_from(ASR_MODEL_PATH)
asr_model.eval()

# Optimised decoding: greedy_batch is faster than greedy for single files too
# because it uses the same batched CUDA path.
# ngram_lm_alpha=0.2 is a light language-model blend — keeps word sequences
# natural without over-correcting.  Set ngram_lm_model=None to skip LM entirely.
decoding_config = {
    'strategy'       : 'greedy_batch',
    'ngram_lm_model' : ASR_LM_PATH,   # pass None here to disable
    'ngram_lm_alpha' : 0.2,
}
asr_model.change_decoding_strategy(decoding_config)

print('✅ English ASR model loaded')

[NeMo W 2026-03-28 14:04:03 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


[NeMo I 2026-03-28 14:04:05 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-28 14:04:05 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-03-28 14:04:05 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-03-28 14:04:06 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /drive/MyDrive/asr_sangam/fastconformer.nemo.
[NeMo I 2026-03-28 14:04:06 ctc_bpe_models:357] Changed decoding strategy to 
    strategy: greedy_batch
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
      ngram_lm_model: null
      ngram_lm_alpha: 0.0
      boosting_tree:
        model_path: null
        key_phrases_file: null
        key_phrases_list: null
        key_phrase_items_list: null
        context_score: 1.0
        depth_scaling

## 🔧 Section 4 — Post-Retrieval Helpers

These are the v4 helpers (over-fetch diversity, recency boost, context ordering)
carried over from the RAG notebook.


In [16]:
# ── Cell 5 : RAG v4 post-retrieval helpers ────────────────────
import math
from datetime     import datetime, timezone
from urllib.parse import urlparse


def apply_recency_boost(results, recency_weight=1.0, half_life_days=30):
    """Add exponential-decay freshness bonus to rerank_score; re-sort descending."""
    now = datetime.now(timezone.utc)
    for r in results:
        try:
            pub = datetime.fromisoformat(str(r.get('date', '')).replace('Z', '+00:00'))
            if pub.tzinfo is None:
                pub = pub.replace(tzinfo=timezone.utc)
        except ValueError:
            pub = datetime(2000, 1, 1, tzinfo=timezone.utc)
        age_days            = max((now - pub).total_seconds() / 86_400, 0)
        recency_factor      = math.exp(-age_days / half_life_days)
        r['recency_factor'] = round(recency_factor, 3)
        r['boosted_score']  = round(r['rerank_score'] + recency_weight * recency_factor, 4)
    return sorted(results, key=lambda x: x['boosted_score'], reverse=True)


def diversify_and_trim(results, top_k, max_per_domain=2):
    """
    Walk the large candidate pool in score order.
    Accept up to max_per_domain chunks per domain; stop at top_k.
    Backfill from bench if not enough diverse sources exist.
    """
    domain_count = {}
    selected, bench = [], []
    for r in results:
        if len(selected) >= top_k:
            break
        domain = urlparse(r.get('url', '')).netloc.lstrip('www.')
        cnt    = domain_count.get(domain, 0)
        if cnt < max_per_domain:
            domain_count[domain] = cnt + 1
            selected.append(r)
        else:
            bench.append(r)
    if len(selected) < top_k:
        selected.extend(bench[:top_k - len(selected)])
    return selected


def order_context_by_recency(results):
    """Sort results newest-first so the SLM reads the freshest article first."""
    def parse_date(r):
        try:
            d = datetime.fromisoformat(str(r.get('date', '')).replace('Z', '+00:00'))
            return d if d.tzinfo else d.replace(tzinfo=timezone.utc)
        except ValueError:
            return datetime(2000, 1, 1, tzinfo=timezone.utc)
    return sorted(results, key=parse_date, reverse=True)


print('✅ Post-retrieval helpers ready')

✅ Post-retrieval helpers ready


## 🌐 Section 5 — Define the FastAPI App

Two inference endpoints:

**`POST /ask`** — accepts a JSON body:
```json
{ "question": "...", "min_cosine": -1, "days_filter": -1, "top_k": -1 }
```
Pass `-1` for any numeric field to use the server default.

**`POST /ask_audio`** — accepts a multipart form upload:
- `audio` — a `.wav` file (required)
- `min_cosine`, `days_filter`, `top_k` — optional floats/ints as form fields

The audio route transcribes the file with the English ASR model,
then feeds the transcription into the same RAG + SLM pipeline as `/ask`.
The response includes both `transcription` and `rag_answer`.


In [17]:
# ── Cell 6 : FastAPI app definition ──────────────────────────
import os
import tempfile
import threading

from fastapi            import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses  import JSONResponse
from pydantic           import BaseModel
from typing             import Optional

from rag.retriever import retrieve
from rag.generator import build_context
from rag.config    import DEFAULT_TOP_K, DEFAULT_DAYS_FILTER, MIN_COSINE
from inference     import run_inference

app = FastAPI(
    title       = 'Nepal RAG + SLM + ASR API',
    description = (
        'Send a text question or an English audio file. '
        'Audio input returns a transcription + RAG-grounded answer. '
        'Text input returns only the RAG-grounded answer.'
    ),
    version='2.0.0',
)

# ── Inference tuning constants ────────────────────────────────
# Adjust here to change server-wide defaults.
MAX_NEW_TOKENS   = 128   # SLM decode length  (raise for longer answers)
FAST_CONTEXT_CAP = 1_200 # chars passed to SLM (raise for richer context)
MAX_PER_DOMAIN   = 2     # chunks per domain in final top_k
RECENCY_WEIGHT   = 1.0   # freshness bonus (0 = pure relevance)
HALF_LIFE_DAYS   = 90    # age in days at which recency bonus is halved
OVERFETCH_FACTOR = 4     # retrieve this many × top_k before diversity filter


# ── Shared RAG + SLM logic ────────────────────────────────────
def run_rag_pipeline(question: str, min_cosine: float, days_filter: int, top_k: int):
    """
    Run the full v4 RAG pipeline:
      over-fetch → recency boost → diversity trim → newest-first context → SLM

    Returns a dict with keys: answer, sources, fallback (bool), params_used.
    This is extracted into a helper so both /ask and /ask_audio reuse it.
    """
    fetch_k    = top_k * OVERFETCH_FACTOR
    candidates = retrieve(question, top_k=fetch_k,
                          days_filter=days_filter, min_cosine=min_cosine)

    if candidates:
        candidates = apply_recency_boost(candidates,
                                         recency_weight=RECENCY_WEIGHT,
                                         half_life_days=HALF_LIFE_DAYS)
        results = diversify_and_trim(candidates,
                                     top_k=top_k,
                                     max_per_domain=MAX_PER_DOMAIN)
    else:
        results = []

    # ── Fallback: no articles → SLM answers from training knowledge ──
    if not results:
        instruction = (
            'You are a helpful assistant. Answer the following question '
            'as accurately and concisely as possible in 2-3 sentences. '
            'If you do not know the answer, say so clearly.'
        )
        input_text = f'Question: {question}'
        with torch.inference_mode():
            answer = run_inference(
                model          = slm_model,
                config         = slm_config,
                instruction    = instruction,
                input_text     = input_text,
                device         = DEVICE,
                max_new_tokens = MAX_NEW_TOKENS,
            )
        return {
            'answer'    : answer,
            'sources'   : [],
            'fallback'  : True,
            'message'   : 'No relevant articles found — answer is from SLM general knowledge.',
            'params_used': {'min_cosine': min_cosine, 'days_filter': days_filter, 'top_k': top_k},
        }

    # ── RAG path ──────────────────────────────────────────────
    context_ordered = order_context_by_recency(results)
    context         = build_context(context_ordered)
    trimmed_context = context[:FAST_CONTEXT_CAP]

    instruction = (
        'Based on the following Nepal news articles (most recent first), '
        'answer the question in 2-3 sentences. '
        'Prioritise information from the most recent article when details conflict.'
    )
    input_text = f'Articles:\n{trimmed_context}\n\nQuestion: {question}'

    with torch.inference_mode():
        answer = run_inference(
            model          = slm_model,
            config         = slm_config,
            instruction    = instruction,
            input_text     = input_text,
            device         = DEVICE,
            max_new_tokens = MAX_NEW_TOKENS,
        )

    sources = [
        {
            'rank'          : i + 1,
            'title'         : r.get('title') or '(no title)',
            'url'           : r.get('url', ''),
            'date'          : r.get('date', ''),
            'domain'        : urlparse(r.get('url', '')).netloc.lstrip('www.'),
            'rerank_score'  : round(r.get('rerank_score', 0), 3),
            'boosted_score' : round(r.get('boosted_score', 0), 3),
            'cosine_score'  : round(r.get('cosine_score', 0), 3),
            'recency_factor': round(r.get('recency_factor', 0), 3),
        }
        for i, r in enumerate(context_ordered)
    ]

    return {
        'answer'    : answer,
        'sources'   : sources,
        'fallback'  : False,
        'params_used': {'min_cosine': min_cosine, 'days_filter': days_filter, 'top_k': top_k},
    }


# ── Request schema for /ask ────────────────────────────────────
class AskRequest(BaseModel):
    question   : str
    min_cosine : Optional[float] = -1   # -1 → server default
    days_filter: Optional[int]   = -1   # -1 → server default
    top_k      : Optional[int]   = -1   # -1 → server default


# ── GET /health ────────────────────────────────────────────────
@app.get('/health', summary='Health check')
async def health():
    """Returns server status, GPU info, and loaded model names."""
    return {
        'status'        : 'ok',
        'device'        : str(DEVICE),
        'cuda_available': torch.cuda.is_available(),
        'gpu_name'      : torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'models_loaded' : {
            'slm'      : SLM_MODEL_PATH,
            'asr'      : ASR_MODEL_PATH,
            'asr_lm'   : ASR_LM_PATH,
        },
        'rag_defaults'  : {
            'min_cosine'      : MIN_COSINE,
            'days_filter'     : DEFAULT_DAYS_FILTER,
            'top_k'           : DEFAULT_TOP_K,
            'overfetch_factor': OVERFETCH_FACTOR,
            'max_per_domain'  : MAX_PER_DOMAIN,
            'recency_weight'  : RECENCY_WEIGHT,
            'half_life_days'  : HALF_LIFE_DAYS,
            'max_new_tokens'  : MAX_NEW_TOKENS,
        },
    }


# ── POST /ask  (text question) ─────────────────────────────────
@app.post('/ask', summary='Text question → RAG answer')
async def ask(req: AskRequest):
    """
    Send a plain-text question. Returns the RAG-grounded SLM answer + sources.
    `transcription` is always null on this endpoint.

    Pass -1 for min_cosine / days_filter / top_k to use server defaults.
    """
    if not req.question.strip():
        raise HTTPException(status_code=400, detail='question must not be empty.')

    min_cosine  = MIN_COSINE          if req.min_cosine  == -1 else req.min_cosine
    days_filter = DEFAULT_DAYS_FILTER if req.days_filter == -1 else req.days_filter
    top_k       = DEFAULT_TOP_K       if req.top_k       == -1 else req.top_k

    try:
        result = run_rag_pipeline(req.question, min_cosine, days_filter, top_k)
        return JSONResponse(content={
            'transcription': None,   # not applicable for text input
            'question'     : req.question,
            **result,
        })
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── POST /ask_audio  (audio file upload) ───────────────────────
@app.post('/ask_audio', summary='English audio → transcription + RAG answer')
async def ask_audio(
    audio      : UploadFile = File(..., description='WAV audio file (English speech)'),
    min_cosine : float      = Form(-1,  description='Cosine similarity threshold (-1 = server default)'),
    days_filter: int        = Form(-1,  description='Max article age in days (-1 = server default)'),
    top_k      : int        = Form(-1,  description='Number of final articles (-1 = server default)'),
):
    """
    Upload an English WAV file. The server:
    1. Transcribes the audio using the FastConformer ASR model.
    2. Feeds the transcription into the RAG + SLM pipeline.
    3. Returns both the transcription and the RAG-grounded answer.

    The `transcription` field in the response contains the recognised English text.
    The `rag_answer` field contains the SLM answer grounded in retrieved articles.
    """
    # ── Validate file type ────────────────────────────────────
    filename  = audio.filename or ''
    extension = os.path.splitext(filename)[-1].lower()
    if extension not in ('.wav', '.wave'):
        raise HTTPException(
            status_code=422,
            detail=f'Only .wav files are supported. Got: "{extension or "(no extension)"}"'
        )

    # ── Save upload to a temp file ────────────────────────────
    # NeMo's transcribe() requires a file path, not a stream.
    # NamedTemporaryFile with delete=False keeps the file alive after close().
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_path = tmp.name
            content  = await audio.read()
            tmp.write(content)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Failed to save audio: {e}')

    # ── Stage 1 : ASR transcription ───────────────────────────
    try:
        # batch_size=1 is correct here — single file per request.
        # The 'greedy_batch' strategy still uses the fast CUDA path at batch=1.
        transcripts = asr_model.transcribe([tmp_path], batch_size=1)

        # NeMo returns either a list of strings (no LM) or a list of
        # Hypothesis objects (with LM). Handle both.
        first = transcripts[0]
        transcription = first.text if hasattr(first, 'text') else str(first)
        transcription = transcription.strip()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'ASR transcription failed: {e}')
    finally:
        # Always clean up the temp file
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    if not transcription:
        raise HTTPException(
            status_code=422,
            detail='ASR returned an empty transcription. Check the audio quality.'
        )

    # ── Stage 2 : RAG + SLM ───────────────────────────────────
    min_cosine_val  = MIN_COSINE          if min_cosine  == -1 else min_cosine
    days_filter_val = DEFAULT_DAYS_FILTER if days_filter == -1 else days_filter
    top_k_val       = DEFAULT_TOP_K       if top_k       == -1 else top_k

    try:
        result = run_rag_pipeline(transcription, min_cosine_val, days_filter_val, top_k_val)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'RAG pipeline failed: {e}')

    # ── Return both transcription and RAG answer ──────────────
    return JSONResponse(content={
        'transcription': transcription,   # ← English ASR output
        'question'     : transcription,   # the transcription IS the question
        **result,
    })


print('✅ FastAPI app defined')
print('   Endpoints: GET /health  |  POST /ask  |  POST /ask_audio  |  GET /docs')

✅ FastAPI app defined
   Endpoints: GET /health  |  POST /ask  |  POST /ask_audio  |  GET /docs


## 🚀 Section 6 — Start uvicorn + ngrok Tunnel

Run this cell last. The server stays alive as long as the cell is running.
Copy the **Public URL** printed below to use from your local PC.

**Quick test from terminal:**
```bash
# Health check
curl https://<your-ngrok-url>/health

# Text question
curl -X POST https://<your-ngrok-url>/ask \
     -H 'Content-Type: application/json' \
     -d '{"question": "when was the protest for the monarchy in Nepal?"}'

# Audio upload
curl -X POST https://<your-ngrok-url>/ask_audio \
     -F 'audio=@/path/to/your/recording.wav'
```


In [18]:
# ── Cell 7 : Start uvicorn + ngrok ───────────────────────────
import threading
import uvicorn
from pyngrok import ngrok

PORT = 8000

tunnel     = ngrok.connect(PORT)
public_url = tunnel.public_url if hasattr(tunnel, 'public_url') else str(tunnel)

print('\n' + '=' * 68)
print('  🚀  Nepal RAG + SLM + ASR — FastAPI Server is LIVE')
print('=' * 68)
print(f'  📡  Public URL      : {public_url}')
print(f'  🩺  Health check    : {public_url}/health')
print(f'  🧠  Text question   : POST {public_url}/ask')
print(f'  🎙️  Audio question  : POST {public_url}/ask_audio')
print(f'  📖  Swagger docs    : {public_url}/docs')
print('=' * 68)
print()
print('  Example (text):')
print(f'    curl -X POST {public_url}/ask \\')
print( '         -H \'Content-Type: application/json\' \\')
print( '         -d \'{"question": "What is happening in Nepal politics?"}\'')
print()
print('  Example (audio):')
print(f'    curl -X POST {public_url}/ask_audio \\')
print( '         -F \'audio=@/path/to/recording.wav\'')
print('=' * 68 + '\n')

config = uvicorn.Config(
    app       = app,
    host      = '0.0.0.0',
    port      = PORT,
    log_level = 'warning',  # change to 'info' to see every request in the output
)
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print('⏳ Server running — keep this cell alive.')
print('   Stop: Runtime → Interrupt execution  (closes the ngrok tunnel too).')


  🚀  Nepal RAG + SLM + ASR — FastAPI Server is LIVE
  📡  Public URL      : https://lyndsay-intraspinal-cayson.ngrok-free.dev
  🩺  Health check    : https://lyndsay-intraspinal-cayson.ngrok-free.dev/health
  🧠  Text question   : POST https://lyndsay-intraspinal-cayson.ngrok-free.dev/ask
  🎙️  Audio question  : POST https://lyndsay-intraspinal-cayson.ngrok-free.dev/ask_audio
  📖  Swagger docs    : https://lyndsay-intraspinal-cayson.ngrok-free.dev/docs

  Example (text):
    curl -X POST https://lyndsay-intraspinal-cayson.ngrok-free.dev/ask \
         -H 'Content-Type: application/json' \
         -d '{"question": "What is happening in Nepal politics?"}'

  Example (audio):
    curl -X POST https://lyndsay-intraspinal-cayson.ngrok-free.dev/ask_audio \
         -F 'audio=@/path/to/recording.wav'

⏳ Server running — keep this cell alive.
   Stop: Runtime → Interrupt execution  (closes the ngrok tunnel too).
